# Sparse Expert AutoSec Training (Colab)
Upload or clone this repository into `/content/That-Ai-Coder` before running.


## Open-source dataset source
This notebook integrates **CVEfixes**: `https://github.com/secureIT-project/CVEfixes`.


In [ ]:
!pip -q install pandas

In [ ]:
import os, pathlib
repo_root = pathlib.Path('/content/That-Ai-Coder')
assert repo_root.exists(), 'Place repository at /content/That-Ai-Coder'
os.chdir(repo_root)


In [ ]:
from pathlib import Path
from sparse_autosec.dataset import OpenSourceDatasetLoader

loader = OpenSourceDatasetLoader(Path('/content/datasets'))
cvefixes_repo = loader.clone_cvefixes_repo()
print('CVEfixes:', cvefixes_repo)
bootstrap = loader.build_bootstrap_samples()
print('bootstrap size', len(bootstrap), 'balance', loader.class_balance(bootstrap))


In [ ]:
from sparse_autosec.system import SparseExpertAutoSec
from sparse_autosec.config import AutoSecConfig

cfg = AutoSecConfig()
cfg.policy.max_fuzz_cases = 8
cfg.execution.fuzz_rounds = 8
cfg.execution.mutation_rounds = 8
system = SparseExpertAutoSec(config=cfg)

for sample in bootstrap:
    task = loader.as_task_text(sample)
    decision = system.router.route(system.core.encode_task(task), complexity=system.core.score_task_complexity(task))
    label = 1.0 if sample.label else 0.0
    system.learning.record_replay(task, decision.selected[0], label)
    if sample.label:
        system.memory.counters[sample.signature] = max(3, system.memory.counters.get(sample.signature, 0))
        if system.learning.enter_training(sample.signature):
            metrics = system.learning.train_cycle(task, decision.selected, success_target=1.0)
            system.learning.leave_training()
            print('trained', sample.signature, metrics['mean_loss'])


In [ ]:
from pathlib import Path
import json
out = Path('/content/artifacts')
out.mkdir(exist_ok=True)
(out / 'adapter_delta.json').write_text(json.dumps(system.core.adapter.delta))
print('saved', out / 'adapter_delta.json')
